# Automatic SWE quality control

UPLA / Glacies brief report, austral winter 2026 (El Niño).

This notebook covers **SWE only** (`min`, `media`, `max`): physical cap, isolated spikes (7-day MAD window), rolling-median smooth.

Site-specific steps that need snow depth (SD alignment, Laguna Negra SD, Quebrada Larga 2026 pillow checks) stay in the main `ElNiño_2026` project (`cleaning_script/`).

`example_data/` has Quebrada Larga and Portillo SWE files. With the full project tree, paths use `Data/Station_SWE/`.

Python 3.10+, pandas, numpy. Matplotlib optional for the plot at the end.


Paths and station settings. Example clone: SWE CSVs in `example_data/`.


In [ ]:
from __future__ import annotations

import shutil
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

PKG = Path.cwd().resolve()
if not (PKG / "example_data").is_dir() and (PKG / "snow_qc_github" / "example_data").is_dir():
    PKG = PKG / "snow_qc_github"
elif not (PKG / "example_data").is_dir() and (PKG.parent / "snow_qc_github" / "example_data").is_dir():
    PKG = PKG.parent / "snow_qc_github"

PROJECT = PKG.parent if (PKG.parent / "Data" / "Station_SWE").is_dir() else None
_EXAMPLE_DIR = PKG / "example_data"
USE_EXAMPLE = _EXAMPLE_DIR.is_dir() and any(_EXAMPLE_DIR.glob("*_SWE.csv"))

if USE_EXAMPLE:
    SWE_DIR = _EXAMPLE_DIR
    BACKUP_DIR = _EXAMPLE_DIR / "_backup"
    OUT_DIR = PKG / "qc_outputs"
else:
    root = PROJECT or PKG.parent
    SWE_DIR = root / "Data" / "Station_SWE"
    BACKUP_DIR = SWE_DIR / "_original_backup"
    OUT_DIR = PKG / "qc_outputs"

OUT_DIR.mkdir(parents=True, exist_ok=True)

STATION_ID: str | None = "04520006" if USE_EXAMPLE else None  # Quebrada Larga; "05401007" = Portillo
WRITE_FILES = not USE_EXAMPLE
MAKE_BACKUP = True

SWE_DEFAULT_CLEAN = {"fwindow": 3, "k": 4.0, "abs_thr_mm": 100.0, "smooth_window": 3}


@dataclass
class StationProfile:
    station_id: str
    name: str
    swe_filename: str
    swe_max_mm: float = 3500.0
    swe_clean: dict[str, Any] = field(default_factory=lambda: dict(SWE_DEFAULT_CLEAN))


STATIONS: dict[str, StationProfile] = {
    "04520006": StationProfile("04520006", "Quebrada Larga", "QUEBRADA_LARGA_SWE.csv", swe_max_mm=3500.0),
    "05401007": StationProfile(
        "05401007", "Portillo", "PORTILLO_SWE.csv", swe_max_mm=1200.0,
        swe_clean={**SWE_DEFAULT_CLEAN, "k": 3.5, "abs_thr_mm": 75.0, "smooth_window": 9},
    ),
    "05703009": StationProfile("05703009", "Laguna Negra", "LAGUNA_NEGRA_SWE.csv", swe_max_mm=3000.0),
    "06020001": StationProfile("06020001", "Termas del Flaco", "TERMAS_DEL_FLACO_SWE.csv", swe_max_mm=1200.0),
    "07301000": StationProfile("07301000", "Lo Aguirre", "LO_AGUIRRE_SWE.csv", swe_max_mm=1200.0),
}

if USE_EXAMPLE:
    print(f"example mode — STATION_ID={STATION_ID!r}")
else:
    print(f"SWE dir: {SWE_DIR}")


SWE cleaning (MAD spikes + smoothing). Parameters also in `QC_PARAMETERS.csv`.


In [ ]:
SWE_COLS = ("min", "media", "max")


def detect_spikes_mad(data, cols, *, fwindow=3, k=4.0, abs_thr=100.0):
    df_clean = data.copy()
    flags = pd.DataFrame(0, index=df_clean.index, columns=cols, dtype="int8")
    win_size = 2 * fwindow + 1
    for col in cols:
        s = df_clean[col].astype(float)
        med = s.rolling(win_size, center=True).median()
        mad = (s - med).abs().rolling(win_size, center=True).median()
        mad_safe = mad.replace(0, np.nan)
        z = (s - med).abs() / mad_safe
        prev_diff = (s - s.shift(1)).abs()
        next_diff = (s - s.shift(-1)).abs()
        neigh_diff = (s.shift(1) - s.shift(-1)).abs()
        safe_prev = prev_diff.fillna(np.inf)
        safe_next = next_diff.fillna(np.inf)
        spike_mad = (
            (mad_safe > 0) & (z > k)
            & (safe_prev > k * mad_safe) & (safe_next > k * mad_safe)
            & (neigh_diff.fillna(0) < k * mad_safe)
        )
        spike_abs = (prev_diff > abs_thr) & (next_diff > abs_thr)
        spike = spike_mad | spike_abs
        df_clean.loc[spike, col] = np.nan
        flags.loc[spike, col] = 1
    return df_clean, flags


def clip_swe_nonnegative(data, cols=None):
    out = data.copy()
    for col in cols or SWE_COLS:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce").clip(lower=0)
    return out


def cap_swe_physical(data, cols, *, max_mm):
    out = clip_swe_nonnegative(data, cols)
    for col in cols:
        s = out[col].astype(float)
        out.loc[s.gt(max_mm), col] = np.nan
    return out


def smooth_swe_series(data, cols, *, window=3, interpolate_limit=2):
    out = data.copy()
    for col in cols:
        s = out[col].astype(float)
        s = s.interpolate(limit=interpolate_limit, limit_direction="both")
        s = s.rolling(window, center=True, min_periods=1).median()
        out[col] = s
    return out


def clean_swe_dataframe(df, *, max_mm, fwindow=3, k=4.0, abs_thr_mm=100.0, smooth_window=3):
    work = df.copy()
    for col in SWE_COLS:
        if col not in work.columns:
            raise ValueError(f"Missing column {col}")
    present = [c for c in SWE_COLS if work[c].notna().any()]
    if not present:
        return work, {c: 0 for c in SWE_COLS}
    work = cap_swe_physical(work, present, max_mm=max_mm)
    cleaned, flags = detect_spikes_mad(work, present, fwindow=fwindow, k=k, abs_thr=abs_thr_mm)
    cleaned = smooth_swe_series(cleaned, present, window=smooth_window)
    cleaned = clip_swe_nonnegative(cleaned, present)
    counts = {col: int(flags[col].sum()) if col in flags.columns else 0 for col in SWE_COLS}
    return cleaned, counts


Run SWE QC.


In [ ]:
def backup_swe_once(path: Path) -> Path:
    BACKUP_DIR.mkdir(parents=True, exist_ok=True)
    dest = BACKUP_DIR / path.name
    if not dest.exists() and path.exists():
        shutil.copy2(path, dest)
        print(f"  backup → {dest}")
    return dest if dest.exists() else path


def run_station_qc(profile: StationProfile) -> dict[str, Any]:
    swe_path = SWE_DIR / profile.swe_filename
    if not swe_path.exists():
        return {"station_id": profile.station_id, "name": profile.name, "error": f"Missing {swe_path}"}

    if MAKE_BACKUP:
        backup_swe_once(swe_path)
    src = BACKUP_DIR / swe_path.name
    if not src.exists():
        src = swe_path
    swe_raw = pd.read_csv(src, parse_dates=["date"])

    kwargs = {**SWE_DEFAULT_CLEAN, **profile.swe_clean}
    swe_clean, spikes = clean_swe_dataframe(swe_raw, max_mm=profile.swe_max_mm, **kwargs)

    out_path = OUT_DIR / profile.swe_filename
    swe_clean.to_csv(out_path, index=False, date_format="%Y-%m-%d")
    if WRITE_FILES:
        swe_clean.to_csv(swe_path, index=False, date_format="%Y-%m-%d")

    return {
        "station_id": profile.station_id,
        "name": profile.name,
        "spikes": spikes,
        "qc_output_swe": str(out_path),
        "wrote_data": WRITE_FILES,
    }


def stations_to_run(selected: str | None) -> list[str]:
    if selected:
        return [selected]
    if USE_EXAMPLE:
        return [s for s, p in STATIONS.items() if (SWE_DIR / p.swe_filename).is_file()]
    return list(STATIONS)


def run_all(selected: str | None = STATION_ID) -> pd.DataFrame:
    rows = []
    for sid in stations_to_run(selected):
        prof = STATIONS[sid]
        print(f"\n=== {prof.name} ({sid}) ===")
        rows.append(run_station_qc(prof))
    return pd.DataFrame(rows)


summary = run_all(STATION_ID)
try:
    from IPython.display import display
    display(summary)
except ImportError:
    print(summary.to_string(index=False))


Optional: raw vs cleaned mean SWE (`media` column).


In [ ]:
try:
    import matplotlib.pyplot as plt
except ImportError:
    print("Install matplotlib: pip install matplotlib")
else:
    diag_id = STATION_ID or "04520006"
    prof = STATIONS[diag_id]
    raw_path = BACKUP_DIR / prof.swe_filename
    if not raw_path.exists():
        raw_path = SWE_DIR / prof.swe_filename
    raw = pd.read_csv(raw_path, parse_dates=["date"])
    clean = pd.read_csv(OUT_DIR / prof.swe_filename, parse_dates=["date"])
    fig, ax = plt.subplots(figsize=(10, 3.5))
    ax.plot(raw["date"], raw["media"], color="#ccc", lw=0.8, label="mean SWE (raw / backup)")
    ax.plot(clean["date"], clean["media"], color="#c45c26", lw=1.0, label="mean SWE (QC)")
    ax.set_title(f"{prof.name} — SWE (media column)")
    ax.set_ylabel("mm")
    ax.legend(loc="upper right", fontsize=8)
    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()
    plt.close(fig)


Full pipeline (SD alignment and site rules) is documented in `../cleaning_script/README.md` and `QC_PARAMETERS.csv`.
